# SkyFinder Temperature Prediction — Training Notebook

Run on **Google Colab with a T4 GPU** (Runtime → Change runtime type → T4 GPU).

1. Mount Drive, clone/pull repo
2. Restore images from the Drive cache made by `01_get_data.ipynb`
3. Fetch metadata + run `prepare_data.py`
4. Metadata-only baseline
5. Smoke test (subset, 1 epoch)
6. Vanilla baseline (15 epochs)
7. Resume (if Colab disconnects) · 8. Results table

In [ ]:
# === Settings ===
STORE_ROOT  = "/content/drive/MyDrive/skyfinder_runs"     # checkpoints, logs, test_preds.csv, test_metrics.json
GITHUB_USER = "alexander-lio"
REPO_NAME   = "imbalanced-regression-on-skyfinder"
IMAGES_DIR  = "/content/images"                           # fast local disk (wiped when the session ends)
IMAGE_TAR   = "/content/drive/MyDrive/skyfinder_cache/skyfinder_images_256.tar"   # made by 01_get_data.ipynb
REPO_DIR    = f"/content/{REPO_NAME}"
!nvidia-smi -L

In [ ]:
# === Mount Drive + clone/pull repo ===
from google.colab import drive
drive.mount("/content/drive")
import os
os.makedirs(STORE_ROOT, exist_ok=True)

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone -q https://github.com/{GITHUB_USER}/{REPO_NAME}.git {REPO_DIR}
os.chdir(REPO_DIR)
!git log --oneline | head -3
!pip install -q tensorboard_logger

In [ ]:
# === Restore images from the Drive cache (~2-5 min) ===
if not os.path.isdir(IMAGES_DIR) or len(os.listdir(IMAGES_DIR)) < 40:
    os.makedirs(IMAGES_DIR, exist_ok=True)
    !tar -xf "{IMAGE_TAR}" -C {IMAGES_DIR}
else:
    print("Images already extracted.")
!ls {IMAGES_DIR} | wc -l
!ls {IMAGES_DIR} | head -5

In [ ]:
# === Metadata CSV + build data/skyfinder.csv ===
!python scripts/download_images.py --metadata_only --metadata data/complete_table_with_mcr.csv
!python scripts/prepare_data.py --metadata data/complete_table_with_mcr.csv --images_dir {IMAGES_DIR} --out data/skyfinder.csv | tail -25
!wc -l data/skyfinder.csv

In [ ]:
# === Metadata-only baseline (no images; camera + month mean) ===
!python scripts/metadata_baseline.py --csv data/skyfinder.csv | tee {STORE_ROOT}/metadata_baseline.txt

## Smoke test
Should finish in a few minutes and print Many / Median / Low and temp-range metrics without errors.
Re-running it needs a new `--store_name` (or delete the old `smoke` folder) because of the overwrite guard.

In [ ]:
!cd dir_skyfinder && python train.py \
    --data_dir {IMAGES_DIR} --csv ../data/skyfinder.csv --store_root {STORE_ROOT} \
    --store_name smoke --subset_frac 0.02 --epoch 1 \
    --lr 1e-4 --batch_size 64 --workers 2 --seed 0

## Vanilla baseline
Watch the time of the first epoch, then multiply by 15. Checkpoints go to Drive every epoch.

In [ ]:
!cd dir_skyfinder && python train.py \
    --data_dir {IMAGES_DIR} --csv ../data/skyfinder.csv --store_root {STORE_ROOT} \
    --store_name vanilla --epoch 15 --lr 1e-4 --schedule 10 13 \
    --batch_size 64 --workers 2 --seed 0

## Resume after a disconnect
Re-run the settings, mount/clone, restore-images and prepare-data cells first, then this one.
`ckpt.pth.tar` is the latest epoch (`ckpt.best.pth.tar` is the best one so far — don't resume from that).

In [ ]:
import glob
RUN = sorted(glob.glob(f"{STORE_ROOT}/skyfinder_resnet50_vanilla_*"))[-1]   # adjust pattern for other runs
print("Resuming:", RUN)
# !cd dir_skyfinder && python train.py \
#     --data_dir {IMAGES_DIR} --csv ../data/skyfinder.csv --store_root {STORE_ROOT} \
#     --store_name vanilla --epoch 15 --lr 1e-4 --schedule 10 13 \
#     --batch_size 64 --workers 2 --seed 0 --resume {RUN}/ckpt.pth.tar

## Results table (all finished runs)

In [ ]:
import json, pathlib, pandas as pd
rows = []
for p in sorted(pathlib.Path(STORE_ROOT).rglob("test_metrics.json")):
    m = json.load(open(p)); tr = m.get("temp_range", {})
    rows.append({"run": p.parent.name,
                 "MAE all": m["overall"]["mae"], "GM all": m["overall"]["gmean"],
                 "MAE many": m["many"]["mae"], "MAE med": m["median"]["mae"], "MAE few": m["low"]["mae"],
                 "MAE <-10C": tr.get("lt_neg10", {}).get("mae"), "MAE >35C": tr.get("gt_35", {}).get("mae")})
pd.DataFrame(rows).round(2)